# TVMaze Content Strategy Data Preparation

## 1. Overview

This notebook prepares the TVMaze dataset for content strategy analysis. The dataset contains TV show information such as genres, languages, ratings, runtime, networks, episode counts, summaries, and release dates.

The analysis focuses on transforming structured API data into a clean and analysis-ready dataset that can support decision-making for a streaming platform. These decisions include understanding which genres are common, which content types are highly rated, how shows differ by language and network, and how content volume can support future acquisition or promotion strategies.

The cleaned dataset produced from this notebook is used for exploratory data analysis, visualization, and business insight generation.

### Dataset Used

The notebook uses the structured TVMaze dataset collected from the API:

- `data/raw/shows_structured.csv`

After cleaning and feature engineering, the final dataset is saved as:

- `data/cleaned/shows_cleaned.csv`

## 2. Import Libraries

The libraries used in this notebook support data loading, cleaning, feature engineering, text processing, visualization, and exploratory analysis.

In [419]:
import os
import re
import requests
import time

import pandas as pd
import warnings
import nltk

import plotly.express as px

from nltk.corpus import stopwords
from textblob import TextBlob
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
from sklearn.exceptions import ConvergenceWarning

print("Libraries imported successfully.")

Libraries imported successfully.


## 3. Load the Dataset

The structured TVMaze dataset is loaded from the raw data folder. This dataset is the starting point for cleaning, feature engineering, and analysis.

In [420]:
file_path = "data/raw/shows_structured.csv"

df = pd.read_csv(file_path)

df.shape

(727, 16)

## 4. Initial Data Inspection

The dataset is inspected to understand its structure, data types, missing values, duplicates, basic statistics, and possible outliers before cleaning.

### 4.1 Dataset Information

This overview shows the dataset structure, column types, and non-null counts.

In [421]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 727 entries, 0 to 726
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   show_id          727 non-null    int64  
 1   name             727 non-null    object 
 2   type             727 non-null    object 
 3   language         727 non-null    object 
 4   genres           669 non-null    object 
 5   status           727 non-null    object 
 6   runtime          680 non-null    float64
 7   average_runtime  726 non-null    float64
 8   premiered        726 non-null    object 
 9   ended            653 non-null    object 
 10  official_site    423 non-null    object 
 11  rating_average   663 non-null    float64
 12  network_name     688 non-null    object 
 13  country_name     688 non-null    object 
 14  country_code     688 non-null    object 
 15  summary          727 non-null    object 
dtypes: float64(3), int64(1), object(12)
memory usage: 91.0+ KB


### 4.2 Missing Values

Missing values are checked to identify which columns need cleaning or special handling.

In [422]:
# Check missing values count and percentage for each column
missing_values = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_summary = pd.DataFrame({
    "missing_count": missing_values,
    "missing_percentage": missing_percentage.round(2)
})

missing_summary = missing_summary[missing_summary["missing_count"] > 0]
missing_summary

,missing_count,missing_percentage
genres,58,7.98
runtime,47,6.46
average_runtime,1,0.14
premiered,1,0.14
ended,74,10.18
official_site,304,41.82
rating_average,64,8.80
network_name,39,5.36
country_name,39,5.36
country_code,39,5.36


### 4.3 Duplicate Rows

In [423]:
pd.DataFrame({
    "check": ["Duplicate rows"],
    "count": [df.duplicated().sum()]
})

,check,count
0,Duplicate rows,0


### 4.4 Statistical Summary

Summary statistics are reviewed for the main numerical columns to understand value ranges and possible unusual values before cleaning.

In [424]:
df[["runtime", "average_runtime", "rating_average"]].describe()

,runtime,average_runtime,rating_average
count,680.000000,726.000000,663.000000
mean,51.201471,50.714876,7.486727
std,18.008483,17.632727,0.813225
min,10.000000,8.000000,3.200000
25%,30.000000,30.000000,7.100000
50%,60.000000,60.000000,7.700000
75%,60.000000,60.000000,8.000000
max,120.000000,120.000000,9.200000


### 4.5 Initial Inspection Findings

The dataset contains 727 rows and 16 columns, with no duplicate rows.

Several columns contain missing values, including `genres`, `runtime`, `ended`, `official_site`, `rating_average`, `network_name`, and country-related fields. These missing values will be handled based on the meaning of each column instead of applying one method to all columns.

The initial inspection also shows that `premiered` and `ended` need to be converted to datetime format, and the `summary` column needs text cleaning because it contains HTML tags.

### 4.6 Outlier Detection

Outlier detection was applied to the main numerical columns: `runtime`, `average_runtime`, and `rating_average`.

The Interquartile Range (IQR) method was used to identify unusually high or low values. Outliers were reviewed but not removed automatically, because some extreme values may represent valid TV show characteristics, such as long runtimes or unusual show formats.

In [425]:
# Outlier detection using the IQR method

outlier_columns = ["runtime", "average_runtime", "rating_average"]

outlier_summary = []

for column in outlier_columns:
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outliers = df[
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    ]

    outlier_summary.append({
        "column": column,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "outlier_count": len(outliers)
    })

outlier_summary_df = pd.DataFrame(outlier_summary)
outlier_summary_df

,column,Q1,Q3,IQR,lower_bound,upper_bound,outlier_count
0,runtime,30.0,60.0,30.0,-15.00,105.00,14
1,average_runtime,30.0,60.0,30.0,-15.00,105.00,12
2,rating_average,7.1,8.0,0.9,5.75,9.35,23


## 5. Data Cleaning Plan

The initial inspection showed missing values, date columns that need conversion, and text fields that need cleaning. The cleaning approach is based on the meaning of each column instead of applying one method to all columns.

The cleaning plan includes:

- Convert `premiered` and `ended` columns to datetime format.
- Remove rows with missing `premiered` values because premiere date is required for time-based analysis.
- Keep missing values in the `ended` column as null because they may indicate that the show is still running.
- Fill missing categorical values in `genres`, `network_name`, `country_name`, and `country_code` with `Unknown`.
- Fill missing `official_site` values with `Not Available`.
- Fill missing numerical values in `runtime`, `average_runtime`, and `rating_average` using the median.
- Remove duplicate rows if any are found.
- Clean the `summary` column by removing HTML tags.
- Prepare the dataset for feature engineering and exploratory analysis.

### 5.1 Irrelevant and Inconsistent Features Review

The dataset columns were reviewed to decide which features are useful for content strategy analysis. Columns such as `type`, `language`, `genres`, `runtime`, `rating_average`, `network_name`, `premiered`, and `summary` were kept because they support the research questions and EDA.

Nested fields from the original API response were already simplified in Step 1 when the raw JSON data was transformed into structured columns. No major irrelevant columns were removed at this stage because the selected columns are relevant to the analysis.

## 6. Data Cleaning

The cleaning plan is applied to prepare the dataset for analysis. A copy of the original dataset is created first, then duplicate rows, date formats, and missing values are handled based on the meaning of each column.

In [426]:
# Create a copy of the dataset for cleaning
df_cleaned = df.copy()

# Remove duplicate rows
df_cleaned = df_cleaned.drop_duplicates()

# Convert date columns to datetime format
df_cleaned["premiered"] = pd.to_datetime(df_cleaned["premiered"], errors="coerce")
df_cleaned["ended"] = pd.to_datetime(df_cleaned["ended"], errors="coerce")

# Remove rows with missing premiere date because it is needed for time-based analysis
df_cleaned = df_cleaned.dropna(subset=["premiered"])

# Fill missing categorical values
categorical_columns = ["genres", "network_name", "country_name", "country_code"]

for column in categorical_columns:
    df_cleaned[column] = df_cleaned[column].fillna("Unknown")

# Fill missing official site values
df_cleaned["official_site"] = df_cleaned["official_site"].fillna("Not Available")

# Fill missing numerical values using median
numerical_columns = ["runtime", "average_runtime", "rating_average"]

for column in numerical_columns:
    df_cleaned[column] = df_cleaned[column].fillna(df_cleaned[column].median())

# Display cleaned dataset shape
pd.DataFrame({
    "stage": ["After cleaning"],
    "rows": [df_cleaned.shape[0]],
    "columns": [df_cleaned.shape[1]]
})

,stage,rows,columns
0,After cleaning,726,16


## 7. Cleaning Verification

The cleaned dataset is checked to confirm that missing values, duplicate rows, and date conversions were handled correctly.

In [427]:
# Check missing values after cleaning
cleaned_missing_values = df_cleaned.isnull().sum()

cleaned_missing_summary = pd.DataFrame({
    "missing_count": cleaned_missing_values
})

cleaned_missing_summary = cleaned_missing_summary[cleaned_missing_summary["missing_count"] > 0]
cleaned_missing_summary

,missing_count
ended,73


In [428]:
pd.DataFrame({
    "check": ["Duplicate rows after cleaning"],
    "count": [df_cleaned.duplicated().sum()]
})

,check,count
0,Duplicate rows after cleaning,0


### 7.1 Cleaning Results

After cleaning, no duplicate rows were found. Missing categorical values were filled with `Unknown`, official website values were filled with `Not Available`, and missing numerical values were filled using the median.

The `premiered` column was converted to datetime format, and rows with missing premiere dates were removed because this column is needed for time-based analysis.

Missing values in the `ended` column were kept as null because they may indicate that the show is still running.

## 8. Feature Engineering

New features are created from the existing columns to support deeper analysis. These features help analyze release trends, show status, genre diversity, episode volume, content clusters, and text information from show summaries.

### 8.1 Feature Engineering Overview

The feature engineering process creates date-based, categorical, text-based, clustering, episode count, and PCA features. These features transform the dataset into a richer format for exploratory analysis and business insight generation.

In [429]:

#  Date-based features 
df_cleaned["premiere_year"] = df_cleaned["premiered"].dt.year
df_cleaned["premiere_month"] = df_cleaned["premiered"].dt.month

# Create a feature to identify whether a show has ended
df_cleaned["is_ended"] = df_cleaned["ended"].notnull()

# Create a feature for show age based on premiere year
current_year = 2026
df_cleaned["show_age"] = current_year - df_cleaned["premiere_year"]

#  Genre features
df_cleaned["genre_count"] = df_cleaned["genres"].apply(
    lambda x: 0 if x == "Unknown" else len(str(x).split(","))
)

#  Text cleaning & counting features 
df_cleaned["summary"] = df_cleaned["summary"].fillna("")
df_cleaned["summary_clean"] = df_cleaned["summary"].apply(
    lambda x: re.sub("<.*?>", "", str(x))
)

df_cleaned["summary_word_count"] = df_cleaned["summary_clean"].apply(
    lambda x: len(str(x).split())
)

df_cleaned["summary_character_count"] = df_cleaned["summary_clean"].apply(
    lambda x: len(str(x))
)

df_cleaned["summary"] = df_cleaned["summary_clean"]
df_cleaned = df_cleaned.drop(columns=["summary_clean"])



### 8.2 Language and Readability Features

In this step, I created simple readability-related features from the cleaned `summary` text. These features help describe the structure and complexity of each show summary.

The created features include sentence count, average sentence length, and average word length. These features can support text-based analysis by showing whether summaries are short, long, simple, or more detailed.

In [430]:

def count_sentences(text):
    text = str(text)
    sentences = re.split(r"[.!?]+", text)
    sentences = [sentence.strip() for sentence in sentences if sentence.strip()]
    return len(sentences)

def average_sentence_length(text):
    text = str(text)
    sentences = re.split(r"[.!?]+", text)
    sentences = [sentence.strip() for sentence in sentences if sentence.strip()]
    
    if len(sentences) == 0:
        return 0
    
    word_counts = [len(sentence.split()) for sentence in sentences]
    return round(sum(word_counts) / len(word_counts), 2)

def average_word_length(text):
    text = str(text)
    words = re.findall(r"\b[a-zA-Z]+\b", text)
    
    if len(words) == 0:
        return 0
    
    word_lengths = [len(word) for word in words]
    return round(sum(word_lengths) / len(word_lengths), 2)

df_cleaned["sentence_count"] = df_cleaned["summary"].apply(count_sentences)
df_cleaned["average_sentence_length"] = df_cleaned["summary"].apply(average_sentence_length)
df_cleaned["average_word_length"] = df_cleaned["summary"].apply(average_word_length)


### 8.3 Keyword Extraction from Summary

Keyword extraction was applied to the cleaned `summary` column to identify important terms that describe each show.

NLTK stopwords were used to remove common English words such as "the", "and", and "is". After removing stopwords, the most frequent meaningful words from each summary were selected as keywords.

In [431]:
# Download NLTK stopwords quietly if not already available
nltk.download("stopwords", quiet=True)

# Load English stopwords from NLTK
stop_words = set(stopwords.words("english"))

def extract_keywords(text, top_n=5):
    text = str(text).lower()
    words = re.findall(r"\b[a-zA-Z]{4,}\b", text)
    meaningful_words = [word for word in words if word not in stop_words]
    
    word_counts = pd.Series(meaningful_words).value_counts()
    return ", ".join(word_counts.head(top_n).index)

df_cleaned["summary_keywords"] = df_cleaned["summary"].apply(extract_keywords)

### 8.4 Sentiment Analysis

TextBlob was used to calculate the sentiment polarity of each cleaned show summary. The sentiment score ranges from -1 to +1, where negative values indicate negative sentiment, positive values indicate positive sentiment, and values close to 0 indicate neutral sentiment.

A new feature called `sentiment_label` was created to classify each summary as Positive, Neutral, or Negative based on the sentiment score.

In [432]:

# Sentiment polarity score: ranges from -1 (very negative) to +1 (very positive)
df_cleaned["sentiment_score"] = df_cleaned["summary"].apply(
    lambda x: TextBlob(str(x)).sentiment.polarity if str(x).strip() != "" else 0
)

# Sentiment label based on score thresholds
df_cleaned["sentiment_label"] = df_cleaned["sentiment_score"].apply(
    lambda x: "Positive" if x > 0.05 else ("Negative" if x < -0.05 else "Neutral")
)
df_cleaned["sentiment_label"].value_counts()



sentiment_label
Positive    432
Neutral     167
Negative    127
Name: count, dtype: int64

### 8.5 Topic Modeling from Summary

To apply topic modeling, I used the cleaned `summary` text to identify general themes in the show descriptions. Since the summaries are short, this step is used as an exploratory text-based feature engineering technique.

The goal is to create a new feature called `summary_topic`, which assigns each show to a general topic based on the words used in its summary.

In [433]:
# Prepare text data from the cleaned summary column
summary_text = df_cleaned["summary"].fillna("")

# Convert summaries into TF-IDF features
tfidf_vectorizer = TfidfVectorizer(
    max_features=500,
    stop_words="english"
)

tfidf_matrix = tfidf_vectorizer.fit_transform(summary_text)

# Apply topic modeling using NMF
num_topics = 5

nmf_model = NMF(
    n_components=num_topics,
    random_state=42,
    max_iter=500
)

topic_matrix = nmf_model.fit_transform(tfidf_matrix)

# Assign the most relevant topic to each show
df_cleaned["summary_topic"] = topic_matrix.argmax(axis=1)

# Display top words for each topic to help interpret the topic numbers
feature_names = tfidf_vectorizer.get_feature_names_out()

for topic_idx, topic in enumerate(nmf_model.components_):
    top_words = [
        feature_names[i]
        for i in topic.argsort()[-10:][::-1]
    ]
    print(f"Topic {topic_idx}: {', '.join(top_words)}")

Topic 0: family, son, year, wife, father, life, american, old, home, business
Topic 1: series, comedy, based, drama, television, american, set, lives, hit, stars
Topic 2: team, police, detective, crime, drama, crimes, solve, criminal, dr, unit
Topic 3: world, years, time, war, earth, mysterious, people, battle, future, story
Topic 4: new, life, city, york, love, women, friends, follows, young, best


The printed topic keywords help interpret the meaning of each topic number. The `summary_topic` column stores the topic number assigned to each show summary.

### 8.6 Rating Category
In this step, a new categorical feature called `rating_category` was created to group shows into Low, Medium, and High categories based on their average rating.

This makes rating values easier to compare and supports later analysis of show quality levels.

In [434]:
df_cleaned["rating_category"] = pd.cut(
    df_cleaned["rating_average"],
    bins=[0, 5, 7, 10],
    labels=["Low", "Medium", "High"],
    include_lowest=True
)

### 8.7 KMeans Clustering and Standardization

KMeans clustering was used as an exploratory technique to group shows into content clusters based on numerical features: `rating_average` and `runtime`.

Before applying KMeans, the selected numerical features were standardized using `StandardScaler`. This is important because KMeans is distance-based, and features with larger numeric ranges can affect the clustering result more strongly.

The new feature `content_cluster` represents the cluster assigned to each show. This feature can help explore whether shows can be grouped into different content segments based on rating and runtime.

In [435]:
# Hide sklearn warnings for cleaner notebook output
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# Select numerical features for clustering
cluster_features = df_cleaned[["rating_average", "runtime"]].copy()

# Standardize features because KMeans is distance-based
scaler = StandardScaler()
cluster_scaled = scaler.fit_transform(cluster_features)

# Apply KMeans clustering
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_cleaned["content_cluster"] = kmeans.fit_predict(cluster_scaled)

# Display cluster distribution
df_cleaned["content_cluster"].value_counts().sort_index()

content_cluster
0    205
1    101
2    420
Name: count, dtype: int64

### 8.8 External Data Integration

The original show-level dataset did not include episode count information, so the TVMaze Episodes API was used to collect the number of episodes for each show.

The collected episode counts were added as a new feature called `episode_count`. This feature helps analyze show length and content volume, which can support content strategy decisions.

After that, `episode_count_category` was created to group shows into Short, Medium, Long, and Very Long categories.

In [436]:
# Collect episode count for each show from the TVMaze Episodes API
episode_counts = []

for show_id in df_cleaned["show_id"]:
    episodes_url = f"https://api.tvmaze.com/shows/{show_id}/episodes"

    try:
        response = requests.get(episodes_url, timeout=10)

        if response.status_code == 200:
            episodes = response.json()
            episode_counts.append(len(episodes))
        else:
            episode_counts.append(0)

    except requests.exceptions.RequestException:
        episode_counts.append(0)

    time.sleep(0.1)

# Add episode count as a new feature
df_cleaned["episode_count"] = episode_counts

# Create episode count category
df_cleaned["episode_count_category"] = pd.cut(
    df_cleaned["episode_count"],
    bins=[-1, 10, 50, 200, float("inf")],
    labels=["Short", "Medium", "Long", "Very Long"]
)

### 8.9 PCA Dimensionality Reduction

PCA was applied as an exploratory dimensionality reduction technique on selected numerical features.

The purpose of PCA is to reduce several numerical variables into two components while keeping part of the original variation in the data. This can help support future visualization, clustering, or modeling.

Before applying PCA, the selected numerical features were standardized using `StandardScaler` because PCA is affected by feature scale.

The two new features, `pca_component_1` and `pca_component_2`, were created as supporting analytical features, not as final business decision variables.

In [437]:
# Select numerical features for PCA
pca_features = df_cleaned[
    [
        "runtime",
        "average_runtime",
        "rating_average",
        "show_age",
        "genre_count",
        "summary_word_count",
        "summary_character_count",
        "episode_count"
    ]
].copy()

# Standardize numerical features before applying PCA
pca_scaler = StandardScaler()
scaled_pca_features = pca_scaler.fit_transform(pca_features)

# Apply PCA with two components
pca = PCA(n_components=2)
pca_components = pca.fit_transform(scaled_pca_features)

# Add PCA components to the dataset
df_cleaned["pca_component_1"] = pca_components[:, 0]
df_cleaned["pca_component_2"] = pca_components[:, 1]

# Display explained variance ratio
pd.DataFrame({
    "PCA Component": ["pca_component_1", "pca_component_2"],
    "Explained Variance Ratio": pca.explained_variance_ratio_
})

,PCA Component,Explained Variance Ratio
0,pca_component_1,0.270215
1,pca_component_2,0.230966


The explained variance ratio shows how much of the original numerical variation is captured by each PCA component. These PCA components were kept as exploratory features for possible visualization or future modeling.

### 8.10 Feature Engineering Preview

After creating the new features, selected columns were previewed to verify that the feature engineering steps were applied correctly.

This preview is kept as the final feature engineering check instead of showing repeated previews after each individual feature.

In [438]:
feature_preview_columns = [
    "name",
    "rating_average",
    "runtime",
    "summary_keywords",
    "sentence_count",
    "average_sentence_length",
    "average_word_length",
    "sentiment_score",
    "sentiment_label",
    "summary_topic",
    "rating_category",
    "content_cluster",
    "episode_count",
    "episode_count_category",
    "pca_component_1",
    "pca_component_2"
]

df_cleaned[feature_preview_columns].head()

,name,rating_average,runtime,summary_keywords,sentence_count,average_sentence_length,average_word_length,sentiment_score,sentiment_label,summary_topic,rating_category,content_cluster,episode_count,episode_count_category,pca_component_1,pca_component_2
0,Under the Dome,6.6,60.0,"dome, town, must, came, answers",2,28.50,4.46,-0.212500,Negative,3,Medium,1,39,Medium,0.060271,0.844451
1,Person of Interest,8.8,60.0,"secret, people, machine, government, every",13,7.38,4.38,-0.404167,Negative,2,High,2,103,Long,0.962790,0.656267
2,Bitten,7.4,60.0,"life, elena, based, thought, found",4,18.75,4.64,0.122727,Positive,4,High,2,33,Medium,0.546320,0.655328
3,Arrow,7.4,60.0,"former, oliver, lance, skills, flame",3,28.33,5.08,-0.055556,Negative,2,High,2,170,Long,0.717513,0.530138
4,True Detective,8.1,60.0,"true, detective, darkness, touch, cast",4,12.00,5.06,0.020455,Neutral,2,High,2,30,Medium,0.106238,1.268488


### 8.11 Feature Engineering Summary

Feature engineering was applied to create new variables that support deeper analysis of the TV show dataset.

The created features include date-based features, text-based features, sentiment analysis features, topic modeling, rating categories, clustering results, episode count features, and PCA components.

These features helped transform the dataset from basic show information into a richer dataset that can support EDA, content strategy analysis, and future modeling.

## 9. Exploratory Data Analysis (EDA)
In this section, the cleaned dataset is explored using statistical summaries and visualizations. The goal is to identify patterns, trends, and initial insights that can support content strategy decisions for the streaming platform.

### 9.1 Distribution of Show Types

This analysis shows the distribution of show types in the dataset. It helps identify which content formats are most available in the TV show catalog.

In [439]:
# Count show types
show_type_counts = df_cleaned["type"].value_counts().reset_index()
show_type_counts.columns = ["type", "count"]

# Plot distribution of show types using Plotly
fig = px.bar(
    show_type_counts,
    x="type",
    y="count",
    title="Distribution of Show Types",
    labels={
        "type": "Show Type",
        "count": "Number of Shows"
    }
)

fig.update_layout(
    xaxis_tickangle=-45,
    width=800,
    height=500
)

fig.show()

### Insight

The chart shows that Scripted shows are the most common type in the dataset, while other types such as Reality, Animation, Documentary, and Talk Show appear much less frequently.

This indicates that the dataset is strongly concentrated around scripted content. Therefore, content strategy recommendations should consider this imbalance and avoid assuming that all show types are equally represented.

### 9.2 Distribution of Languages

This analysis shows the distribution of show languages in the dataset. It helps identify whether the catalog is diverse or concentrated around specific languages.

In [440]:
# Count languages
language_counts = df_cleaned["language"].value_counts().reset_index()
language_counts.columns = ["language", "count"]

# Plot distribution of languages using Plotly
fig = px.bar(
    language_counts,
    x="language",
    y="count",
    title="Distribution of Show Languages",
    labels={
        "language": "Language",
        "count": "Number of Shows"
    }
)

fig.update_layout(
    xaxis_tickangle=-45,
    width=800,
    height=500
)

fig.show()

### Insight

The chart shows that English-language shows dominate the dataset, while Japanese shows appear in a much smaller number.

This indicates a strong language imbalance in the collected data. Therefore, content strategy recommendations should not assume that the dataset represents all languages equally, and additional international content data may be needed for more balanced analysis.

### 9.3 Most Common Genres

This analysis shows the most frequent genres in the dataset. Since each show can have multiple genres, the `genres` column was split so each genre could be counted separately.

In [441]:
# Split genres and count each genre separately
genre_counts = (
    df_cleaned["genres"]
    .str.split(",")
    .explode()
    .str.strip()
    .value_counts()
    .head(10)
    .reset_index()
)

genre_counts.columns = ["genre", "count"]

# Plot top 10 most common genres using Plotly
fig = px.bar(
    genre_counts,
    x="genre",
    y="count",
    title="Top 10 Most Common Genres",
    labels={
        "genre": "Genre",
        "count": "Number of Shows"
    }
)

fig.update_layout(
    xaxis_tickangle=-45,
    width=900,
    height=500
)

fig.show()

### Insight

Drama is the most common genre in the dataset, followed by Comedy, Action, Crime, and Science-Fiction. This shows that the catalog is strongly focused on scripted entertainment genres.

The presence of Unknown among the top genres indicates that some shows did not have genre information available in the original API data. This should be considered when making content strategy decisions, because missing genre data may affect the accuracy of genre-based recommendations.

### 9.4 Average Rating by Genre

This analysis calculates the average rating for each genre. Since each show can belong to multiple genres, the `genres` column was split so each genre could be analyzed separately.

This first view shows the top genres by average rating without applying a minimum show count filter.

In [442]:
# Create a genre-level dataframe for rating analysis
genre_rating_df = df_cleaned.copy()

genre_rating_df["genres"] = genre_rating_df["genres"].str.split(",")
genre_rating_df = genre_rating_df.explode("genres")
genre_rating_df["genres"] = genre_rating_df["genres"].str.strip()

# Calculate average rating by genre
average_rating_by_genre = (
    genre_rating_df
    .groupby("genres")["rating_average"]
    .mean()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

average_rating_by_genre.columns = ["genre", "average_rating"]

In [443]:


# Plot top 10 genres by average rating using Plotly
fig = px.bar(
    average_rating_by_genre,
    x="genre",
    y="average_rating",
    title="Top 10 Genres by Average Rating",
    labels={
        "genre": "Genre",
        "average_rating": "Average Rating"
    }
)

fig.update_layout(
    xaxis_tickangle=-45,
    width=900,
    height=500
)

fig.show()

### Insight

Some genres appear to have high average ratings, but this result should be interpreted carefully because some of these genres may have only a small number of shows.

For more reliable content strategy decisions, average rating should be considered together with genre availability. A genre with a high rating but very few shows may not provide enough evidence for a strong business recommendation.

#### 9.4.1 Average Rating by Genre with Minimum Show Count

To make the rating comparison more reliable, genres were filtered to include only genres with at least 10 shows. This helps reduce the effect of very small genre categories on the average rating.

In [444]:
# Calculate average rating and show count for each genre
genre_stats = (
    genre_rating_df
    .groupby("genres")
    .agg(
        average_rating=("rating_average", "mean"),
        show_count=("show_id", "count")
    )
    .sort_values(by="average_rating", ascending=False)
)

# Keep only genres with at least 10 shows
reliable_genre_stats = genre_stats[genre_stats["show_count"] >= 10].head(10).reset_index()

# Display the reliable genre statistics
reliable_genre_stats

,genres,average_rating,show_count
0,Medical,7.864706,17
1,Crime,7.844030,134
2,Mystery,7.797778,45
3,War,7.792308,13
4,Adventure,7.721176,85
5,History,7.705000,20
6,Science-Fiction,7.696396,111
7,Drama,7.689106,358
8,Action,7.660959,146
9,Thriller,7.643284,67


In [445]:
# Plot top genres by average rating after applying minimum show count
fig = px.bar(
    reliable_genre_stats,
    x="genres",
    y="average_rating",
    title="Top 10 Genres by Average Rating (Minimum 10 Shows)",
    labels={
        "genres": "Genre",
        "average_rating": "Average Rating"
    },
    hover_data=["show_count"]
)

fig.update_layout(
    xaxis_tickangle=-45,
    width=900,
    height=500
)

fig.show()

### Insight

After filtering genres to include only those with at least 10 shows, Medical, Crime, Mystery, War, and Adventure appear among the highest-rated genres.

This makes the comparison more reliable because it avoids making decisions based on genres with very few shows. Although Drama and Action are among the most common genres, some less frequent genres have slightly higher average ratings, which may be useful for future content acquisition decisions.

### 9.5 Relationship Between Runtime and Rating

This analysis explores whether there is a relationship between show runtime and average rating. It helps understand if longer or shorter shows tend to receive higher ratings.

In [446]:
# Plot relationship between runtime and average rating using Plotly
fig = px.scatter(
    df_cleaned,
    x="runtime",
    y="rating_average",
    title="Relationship Between Runtime and Average Rating",
    labels={
        "runtime": "Runtime (minutes)",
        "rating_average": "Average Rating"
    },
    opacity=0.6
)

fig.update_layout(
    width=850,
    height=500
)

fig.show()

In [447]:
runtime_rating_corr = df_cleaned["runtime"].corr(df_cleaned["rating_average"])

pd.DataFrame({
    "Relationship": ["Runtime and Average Rating"],
    "Correlation": [round(runtime_rating_corr, 3)]
})

,Relationship,Correlation
0,Runtime and Average Rating,0.031


### Insight

The correlation between runtime and average rating is 0.031, which indicates a very weak relationship.

This suggests that runtime is not a strong indicator of show rating in this dataset. Therefore, content strategy decisions should not rely on runtime alone when evaluating show quality or potential value.

### 9.6 Show Production Trends Over Time

This analysis shows how the number of premiered shows changed over time. It helps identify content release trends and whether the catalog is concentrated in specific years.

In [448]:
# Count shows by premiere year
shows_by_year = (
    df_cleaned["premiere_year"]
    .value_counts()
    .sort_index()
    .reset_index()
)

shows_by_year.columns = ["premiere_year", "count"]

# Plot show production trends over time using Plotly
fig = px.line(
    shows_by_year,
    x="premiere_year",
    y="count",
    markers=True,
    title="Number of Shows by Premiere Year",
    labels={
        "premiere_year": "Premiere Year",
        "count": "Number of Shows"
    }
)

fig.update_layout(
    width=900,
    height=500
)

fig.show()

### Insight

The number of premiered shows increased gradually over time, especially after the early 2000s. The dataset shows a strong peak around 2014, which suggests that many shows in this sample premiered during that period.

The drop after the peak may not necessarily mean that TV show production decreased overall. It may be related to how the API data was collected, the selected pages, or the availability of records in the dataset. Therefore, this trend should be interpreted carefully.

### 9.7 Top Networks by Number of Shows

This analysis identifies the networks with the highest number of shows in the dataset. It helps understand which networks contribute the most content to the catalog.

In [449]:
# Count top networks
network_counts = (
    df_cleaned["network_name"]
    .value_counts()
    .head(10)
    .reset_index()
)

network_counts.columns = ["network", "count"]

# Plot top networks using Plotly
fig = px.bar(
    network_counts,
    x="network",
    y="count",
    title="Top 10 Networks by Number of Shows",
    labels={
        "network": "Network",
        "count": "Number of Shows"
    }
)

fig.update_layout(
    xaxis_tickangle=-45,
    width=900,
    height=500
)

fig.show()

### Insight

NBC, ABC, CBS, FOX, and HBO are among the top networks by number of shows in the dataset. This suggests that the dataset is strongly represented by major TV networks, especially large U.S. networks.

The presence of `Unknown` among the top values shows that some records did not include network information in the original API data. This should be considered when analyzing network-level performance or making content acquisition decisions.

### 9.8 Rating Category Distribution

This analysis shows how shows are distributed across Low, Medium, and High rating categories. It helps understand the overall quality level of the catalog based on average ratings.

In [450]:
# Count rating categories
rating_category_counts = (
    df_cleaned["rating_category"]
    .value_counts()
    .reset_index()
)

rating_category_counts.columns = ["rating_category", "count"]

# Plot rating category distribution using Plotly
fig = px.bar(
    rating_category_counts,
    x="rating_category",
    y="count",
    title="Distribution of Rating Categories",
    labels={
        "rating_category": "Rating Category",
        "count": "Number of Shows"
    }
)

fig.update_layout(
    width=800,
    height=500
)

fig.show()

### Insight

Most shows in the dataset fall under the High rating category, while fewer shows are classified as Medium, and only a small number are classified as Low. This suggests that the catalog is generally concentrated around highly rated shows.

However, this result should be interpreted carefully because missing ratings were filled using the median. This may increase the number of shows in the higher or middle rating categories and may affect the true distribution of ratings.

### 9.9 Distribution of Episode Count Categories

This analysis shows how shows are distributed across episode count categories. It helps understand whether the catalog is dominated by short or long-running shows, which can support content acquisition decisions.

In [451]:
# Count episode count categories
episode_category_counts = (
    df_cleaned["episode_count_category"]
    .value_counts()
    .reset_index()
)

episode_category_counts.columns = ["episode_count_category", "count"]

# Plot episode count category distribution using Plotly
fig = px.bar(
    episode_category_counts,
    x="episode_count_category",
    y="count",
    title="Distribution of Episode Count Categories",
    labels={
        "episode_count_category": "Episode Count Category",
        "count": "Number of Shows"
    }
)

fig.update_layout(
    width=800,
    height=500
)

fig.show()

### Insight

Medium and Long shows are the most common in the dataset, while Short and Very Long shows appear less frequently. This suggests that the catalog is mainly concentrated around shows with moderate to high episode counts.

For the streaming platform, this may indicate stronger content continuity and better potential for longer viewer engagement.

## 10. Bias and Fairness Evaluation

Bias and fairness are important because the dataset may not represent all groups equally. If the dataset is biased, the insights and recommendations may also be biased.

### 10.1 Common Sources of Data Bias

The main types of bias considered in this project are:

- **Representation bias:** Some groups may be overrepresented or underrepresented.
- **Measurement bias:** Some values may not be measured equally across all records.
- **Historical bias:** The dataset may reflect historical availability patterns.
- **Missing data bias:** Missing values may affect some groups more than others.

### 10.2 Dataset Bias Evaluation

#### Representation Bias

The dataset is strongly concentrated around English-language shows and scripted content. This means that non-English shows and other show types, such as Reality, Documentary, Talk Show, and News, are underrepresented.

As a result, the analysis may favor English scripted content more than other types of shows.

#### Measurement Bias

Some columns contain missing values, such as `official_site`, `rating_average`, `network_name`, and `genres`. Ratings may also not be available for all shows equally, which can affect comparisons between genres, networks, or content types.

#### Historical Bias

The dataset reflects the shows available in the TVMaze API and the selected pages collected. Therefore, it may reflect availability patterns in the collected sample rather than the full TV market.

#### Missing Data Bias

Missing values may affect the analysis if they are not distributed evenly across all shows. For example, shows without ratings, networks, or country information may be excluded from some comparisons or grouped under `Unknown`.

### 10.3 Impact on Results

These biases may affect the final insights and recommendations. For example, the platform may appear to benefit more from investing in English scripted shows because these shows are highly represented in the dataset.

However, this does not necessarily mean that non-English or less common content types are less valuable. Therefore, recommendations should be interpreted carefully and supported by additional data before making final business decisions.

### 10.4 Recommendations to Mitigate Bias

To reduce bias in future analysis, the following steps are recommended:

- Collect more data from additional TVMaze API pages or endpoints.
- Include more non-English shows and international content.
- Compare results across different languages, countries, networks, and content types.
- Avoid making recommendations based only on categories with very small sample sizes.
- Clearly document missing data and avoid over-interpreting incomplete fields.

## 11. Saving the Final Cleaned Dataset

After completing data cleaning, feature engineering, exploratory data analysis, and bias evaluation, the final cleaned dataset is saved as a CSV file. This file will be used for future analysis, visualization, and reporting.

In [452]:
os.makedirs("data/cleaned", exist_ok=True)

# Save the cleaned and feature-engineered dataset
df_cleaned.to_csv("data/cleaned/shows_cleaned.csv", index=False)

print("Final dataset saved:", df_cleaned.shape)

Final dataset saved: (726, 36)


## 12. Final Summary

In Step 2, the TVMaze dataset was inspected, cleaned, enriched with new features, explored through EDA, and reviewed for potential bias and fairness concerns. The goal was to prepare an analysis-ready dataset that can support content strategy decisions for the streaming platform.

### Data Cleaning Summary

The dataset was checked for missing values, duplicate rows, inconsistent values, and data type issues. No duplicate rows were found.

Missing categorical values were filled with `Unknown`, missing official website values were filled with `Not Available`, and missing numerical values were filled using the median. Rows with missing `premiered` values were removed because premiere date is required for time-based analysis. Missing values in the `ended` column were kept as null because they may indicate that the show is still running.

The `summary` column was cleaned by removing HTML tags before applying text-based feature engineering.

### Feature Engineering Summary

New features were created to support deeper analysis, including date-based features, text-based features, sentiment analysis, topic modeling, rating categories, KMeans clustering, episode count features, and PCA components.

These features helped transform the dataset from basic show information into a richer dataset for EDA, content strategy analysis, and possible future modeling.

### EDA Summary

The exploratory data analysis showed that most shows are `Scripted`, and the dataset is highly concentrated around English-language shows. Drama, Comedy, Action, Crime, and Science-Fiction are among the most common genres.

After filtering genres to include only those with at least 10 shows, Medical, Crime, Mystery, War, and Adventure appeared among the highest-rated genres. Runtime showed a very weak relationship with average rating, and the number of premiered shows increased over time with a noticeable peak around 2014.

NBC, ABC, CBS, FOX, and HBO were among the top networks by number of shows. Most shows fell under the `High` rating category, while Medium and Long shows were the most common episode count categories.

### Bias and Fairness Summary

The dataset may contain representation bias because English-language shows and scripted content are strongly represented, while non-English shows and other content types are underrepresented.

Measurement and missing data bias may also exist because some fields, such as ratings, genres, networks, and official websites, contained missing values. Historical bias may be present because the dataset reflects the selected TVMaze API pages rather than the full TV market.

### Final Output

The final cleaned and feature-engineered dataset was saved as:

- `data/cleaned/shows_cleaned.csv`

### Final Note

The results should be interpreted carefully because the dataset is based on a collected sample from the TVMaze API. Additional data from more pages, endpoints, languages, countries, and content types would make the analysis more complete and reduce bias in future recommendations.